In [1]:
from collections import Counter
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
import os

In [2]:
# Load labels and pockets
root = '.'
PATH_TO_PROBS = os.path.join(root, "..", "processed", "unidock_docking", "inference_probs")
labels = np.array(pickle.load(open(os.path.join(PATH_TO_PROBS, "success_mols.pkl"), "rb")))
pockets = [i.replace("_bin_01.npz", "") for i in sorted(os.listdir(PATH_TO_PROBS)) if i != "success_mols.pkl"]
success_mols = np.array(pickle.load(open(os.path.join(PATH_TO_PROBS, "success_mols.pkl"), "rb")))
assert (success_mols == labels).all()

print(f"Number of pockets: {len(pockets)}")
print(f"Number of unique molecules: {len(set(labels))}")

# Load probabilities
print("Loading probabilities...")
pocket_to_all_probs = {}
for c, pocket in tqdm(enumerate(pockets)):
    probs = np.load(os.path.join(PATH_TO_PROBS, f"{pocket}_bin_01.npz"))["arr_0"]
    pocket_to_all_probs[pocket] = probs

# Prepare matrix
M = np.column_stack([pocket_to_all_probs[p] for p in pockets])
del pocket_to_all_probs
print(f"Matrix shape: {M.shape}")
print("Normalizing by columns...")
means_1 = np.mean(M, axis=0, keepdims=True)
stds_1 = np.std(M, axis=0, keepdims=True)
M -= means_1
M /= stds_1

print("Normalizing by rows...")
means_2 = np.mean(M, axis=1, keepdims=True)
stds_2 = np.std(M, axis=1, keepdims=True)
M -= means_2
M /= stds_2

print(f"Matrix loaded, shape: {M.shape}")

Number of pockets: 276
Number of unique molecules: 9556875
Loading probabilities...


276it [00:56,  4.87it/s]


Matrix shape: (9556875, 276)
Normalizing by columns...
Normalizing by rows...
Matrix loaded, shape: (9556875, 276)


In [3]:
# Load smiles
df = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')

# From compound ID to SMILES
synthons = {i: j for i, j in zip(df['id'], df['smiles'])}

# Dictionary with counts of each synthon
syns_counter = dict(Counter([j for i in synthons for j in i.split("____")[1:]]))

print(f"Number of unique 'synthons': {len(syns_counter)}")
print(f"1 occ: {len([i for i in syns_counter if syns_counter[i] == 1])}")
print(f">100 occ: {len([i for i in syns_counter if syns_counter[i] > 100])}")
print(f">1000 occ: {len([i for i in syns_counter if syns_counter[i] > 1000])}")
print(f"MAX occ: {max(syns_counter.values())}")

del df, synthons

Number of unique 'synthons': 1299755
1 occ: 350697
>100 occ: 32236
>1000 occ: 2806
MAX occ: 22418


In [4]:
# Get indices that sort every column
inds = {}
for col in tqdm(range(M.shape[1])):
    inds[col] = np.argsort(M[:, col])[::-1].astype(np.int32)

# Remove huge matrix M from memory
del M

100%|██████████| 276/276 [03:35<00:00,  1.28it/s]


In [5]:
MAX_MOL = 100000
MAX_SYN = 5

ALL_CONSIDERED_MOLECULES = {}
ALL_CONSIDERED_SYNTHONS = {}

for c1, pocket in enumerate(pockets):

    # Molecules sorted
    sorted_molecules = labels[inds[c1]]

    # Counting synthons in the sorted molecules
    syns_counter_tmp = {i: 0 for i in syns_counter}
    considered_molecules = []

    # For each molecule
    for c2, mol in enumerate(sorted_molecules):

        syns = mol.split("____")[1:]
        include = [syns_counter_tmp[syn] < MAX_SYN for syn in syns]

        # If we can include the molecule
        if all(include) == True:
            considered_molecules.append(mol)
            for syn in syns:
                syns_counter_tmp[syn] += 1

        # If we can not include the molecule
        else:
            continue

        # If we have reached MAX_MOL
        if len(considered_molecules) >= MAX_MOL:
            break
    
    # Sanity check
    syns = [syn for mol in considered_molecules for syn in mol.split("____")[1:]]
    syns = dict(Counter(syns))
    assert all([syns[i] <= MAX_SYN for i in syns])

    print("\n")
    print(f"Pocket model: {pocket}")
    print(f"Number of considered molecules: {len(considered_molecules)}")
    print(f"Number of evaluated molecules: {c2+1}")
    print(f"Number of considered synthons: {len(syns)}")

    ALL_CONSIDERED_MOLECULES[pocket] = set(considered_molecules)
    ALL_CONSIDERED_SYNTHONS[pocket] = set(syns)



Pocket model: alphafold2_P9WFS9_model_0_pocket_1
Number of considered molecules: 100000
Number of evaluated molecules: 195204
Number of considered synthons: 138439


Pocket model: alphafold2_P9WFS9_model_0_pocket_2
Number of considered molecules: 100000
Number of evaluated molecules: 278870
Number of considered synthons: 132244


Pocket model: alphafold2_P9WFS9_model_0_pocket_3
Number of considered molecules: 100000
Number of evaluated molecules: 237163
Number of considered synthons: 130375


Pocket model: alphafold2_P9WFS9_model_0_pocket_4
Number of considered molecules: 100000
Number of evaluated molecules: 439101
Number of considered synthons: 135897


Pocket model: alphafold2_P9WFS9_model_0_pocket_5
Number of considered molecules: 100000
Number of evaluated molecules: 156386
Number of considered synthons: 114163


Pocket model: alphafold2_P9WFT1_model_0_pocket_1
Number of considered molecules: 100000
Number of evaluated molecules: 311910
Number of considered synthons: 131866


Po

In [6]:
len(ALL_CONSIDERED_MOLECULES), len(ALL_CONSIDERED_SYNTHONS)

(276, 276)

In [9]:
active_molecules = set([j for pocket in ALL_CONSIDERED_MOLECULES for j in ALL_CONSIDERED_MOLECULES[pocket]])
active_synthons = set([j for pocket in ALL_CONSIDERED_SYNTHONS for j in ALL_CONSIDERED_SYNTHONS[pocket]])

print(f"Molecules being active at least once: {len(active_molecules)}")
print(f"Synthons being active at least once: {len(active_synthons)}")

Molecules being active at least once: 2907538
Synthons being active at least once: 1053829


In [34]:
INACTIVES = set()
for molecule in tqdm(labels):
    # If molecule is not active
    if molecule not in active_molecules:
        # If None of the synthons is in active_synthons
        if any([syn in active_synthons for syn in molecule.split("____")[1:]]) == False:
            INACTIVES.add(molecule)

100%|██████████| 9556875/9556875 [00:11<00:00, 855225.96it/s]


In [38]:
INACTIVES

{np.str_('s_63____7285448____12289816'),
 np.str_('m_272430____26330430____26404864'),
 np.str_('m_276436____14080264____15265352'),
 np.str_('s_273464____11199438____12427488'),
 np.str_('s_273464____11196900____11235082'),
 np.str_('s_265282____9208120____4272282'),
 np.str_('m_40____23053792____21848850'),
 np.str_('s_1500____554082____494340'),
 np.str_('s_273910____11993298____25422804'),
 np.str_('s_273610____11520616____12549026'),
 np.str_('m_274370____25845602____25888964'),
 np.str_('s_272126____10391624____10347512'),
 np.str_('m_271362____11375050____15561612'),
 np.str_('s_62____14048794____7080742'),
 np.str_('m_282512____24295892____24205082'),
 np.str_('m_273496____12229462____17606888'),
 np.str_('m_269862____26653002____27421684'),
 np.str_('m_87____23189302____23186892'),
 np.str_('s_1626____22055424____22132492'),
 np.str_('s_63____15345548____9452858'),
 np.str_('m_2430____22526394____14472848'),
 np.str_('s_271948____17628344____13148592'),
 np.str_('m_273450____1